### 1. For cut

The evaluation protocol maintains the same number of trials and the same target/non-target ratio as the original test list. When the original utterances cannot be mapped to valid segments after duration truncation, replacement trials are generated using speaker-aware random sampling to preserve the statistical properties of the evaluation set.

In [ ]:
import pandas as pd
import random
from pathlib import Path
from collections import defaultdict


random.seed(42)

In [13]:
def load_metadata(metadata_path):
    df = pd.read_csv(metadata_path)
    seg_map = defaultdict(list)
    speaker_map = {}
    speaker_segments = defaultdict(list)

    for _, row in df.iterrows():
        fname = row["filename"]
        spk = row["speaker_id"]
        stem = Path(fname).stem

        original = stem.split("_")[0]
        seg_map[original].append(fname)
        speaker_map[fname] = spk
        speaker_segments[spk].append(fname)
    return seg_map, speaker_map, speaker_segments

def generate_test_list(original_test_list, metadata_path, output_path):
    seg_map, speaker_map, speaker_segments = load_metadata(metadata_path)
    with open(original_test_list) as f:
        trials = []
        with open(original_test_list) as f:
            for line in f:
                line = line.strip().replace('"','')
                parts = line.split()

                if len(parts) != 3:
                    continue

                trials.append(parts)
    target_size = len(trials)
    pairs = []

    # keep label ratio
    label_counts = {"0": 0, "1": 0}
    for label, a, b in trials:
        a_id = Path(a).stem
        b_id = Path(b).stem

        seg_a = seg_map.get(a_id, [])
        seg_b = seg_map.get(b_id, [])

        if seg_a and seg_b: # both A and B have segments
            sa = random.choice(seg_a)
            sb = random.choice(seg_b)
            pairs.append((label, sa, sb))
            label_counts[label] += 1
        elif seg_a: # Only A has segments
            if label == "1" and len(seg_a) >= 2:
                sa, sb = random.sample(seg_a, 2)
            else:
                sa = sb = random.choice(seg_a)
            pairs.append((label, sa, sb))
            label_counts[label] += 1
        elif seg_b: # Only B
            if label == "1" and len(seg_b) >= 2:
                sa, sb = random.sample(seg_b, 2)
            else:
                sa = sb = random.choice(seg_b)
            pairs.append((label, sa, sb))
            label_counts[label] += 1
        else: # Neither
            continue

    # Pool sampling by speaker when not get enough samples (= number of samples of the original test_list)
    speakers = list(speaker_segments.keys())

    while len(pairs) < target_size:
        # keep ratio
        label = random.choice(["0", "1"])
        if label == "1":
            spk = random.choice(speakers)
            segs = speaker_segments[spk]
            if len(segs) < 2:
                continue
            sa, sb = random.sample(segs, 2)
        else:
            spk1, spk2 = random.sample(speakers, 2)
            sa = random.choice(speaker_segments[spk1])
            sb = random.choice(speaker_segments[spk2])
        pairs.append((label, sa, sb))

    pairs = pairs[:target_size]

    with open(output_path, "w") as f:
        for label, a, b in pairs:
            f.write(f"{label}\twav/{a}\twav/{b}\n")
    print("Generated:", len(pairs))

In [ ]:
import os

dir = r"test_data\test_set_O"
durations = [3, 5, 7]

for d in durations:
    meta_path = os.path.join(dir, f"metadata_{d}s.csv")
    test_list_path = os.path.join(dir, "test_list_gt.csv")
    output_path = os.path.join(dir, f"test_list_gt_{d}s.csv")

    generate_test_list(
        test_list_path,
        meta_path,
        output_path
    )

Generated: 9895
Generated: 9895
Generated: 9895


### 2. For train_vi

In [1]:
import os

def generate_test_list_vi(folder, ori_test_list, out_test_list):
    wav_files = set(
        f for f in os.listdir(folder)
        if f.lower().endswith(".wav")
    )
    print(f"Found {len(wav_files)} wav")

    kept = 0
    removed = 0
    new_lines = []

    with open(ori_test_list, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip().replace('"', '')
            parts = line.split()
            if len(parts) != 3:
                continue
            label, a, b = parts
            file_a = os.path.basename(a)
            file_b = os.path.basename(b)

            if file_a in wav_files and file_b in wav_files:
                new_lines.append(f"{label}\twav/{file_a}\twav/{file_b}")
                kept += 1
            else:
                removed += 1

    with open(out_test_list, "w", encoding="utf-8") as f:
        f.write("\n".join(new_lines))

    print("-" * 30)
    print("Pairs kept:", kept)
    print("Pairs removed:", removed)

In [2]:
wav_vi_folder = r"D:\my_project\SLP301-data\test_data\test_set_O\wav_vi"
original_test_list = r"D:\my_project\SLP301-data\test_data\test_set_O\test_list_gt.csv"
output_test_list = r"D:\my_project\SLP301-data\test_data\test_set_O\test_list_gt_vi.csv"

generate_test_list_vi(wav_vi_folder, original_test_list, output_test_list)

Found 16561 wav
------------------------------
Pairs kept: 7148
Pairs removed: 2747


In [7]:
def check_test_list(test_list, wav_folder):
    total = 0
    same = 0
    diff = 0
    missing_files = 0
    duplicate_pairs = 0
    self_pairs = 0
    seen_pairs = set()

    with open(test_list, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip().replace('"', '')
            parts = line.split()
            if len(parts) != 3:
                continue
            label, a, b = parts
            file_a = os.path.basename(a)
            file_b = os.path.basename(b)
            total += 1

            if label == "1":
                same += 1
            else:
                diff += 1

            # check self pair
            if file_a == file_b:
                self_pairs += 1

            # check duplicate
            pair_key = tuple(sorted([file_a, file_b]))
            if pair_key in seen_pairs:
                duplicate_pairs += 1
            else:
                seen_pairs.add(pair_key)

            # check file existence
            path_a = os.path.join(wav_folder, file_a)
            path_b = os.path.join(wav_folder, file_b)
            if not os.path.exists(path_a) or not os.path.exists(path_b):
                missing_files += 1

    print("="*40)
    print("TEST LIST REPORT")
    print("="*40)

    print("Total trials:", total)
    print("Same speaker (1):", same)
    print("Different speaker (0):", diff)

    if total > 0:
        print("Same ratio:", round(same/total, 3))
    print("-"*40)
    print("Missing wav files:", missing_files)
    print("Duplicate pairs:", duplicate_pairs)
    print("Self pairs:", self_pairs)

In [9]:
test_list = r"D:\my_project\SLP301-data\test_data\test_set_O\test_list_gt.csv"
wav_folder = r"D:\my_project\SLP301-data\test_data\test_set_O\wav"
check_test_list(test_list, wav_folder)

TEST LIST REPORT
Total trials: 9895
Same speaker (1): 2446
Different speaker (0): 7449
Same ratio: 0.247
----------------------------------------
Missing wav files: 0
Duplicate pairs: 1002
Self pairs: 0


In [10]:
test_list_vi = r"D:\my_project\SLP301-data\test_data\test_set_O\test_list_gt_vi.csv"
wav_folder_vi = r"D:\my_project\SLP301-data\test_data\test_set_O\wav_vi"
check_test_list(test_list_vi, wav_folder_vi)

TEST LIST REPORT
Total trials: 7148
Same speaker (1): 1151
Different speaker (0): 5997
Same ratio: 0.161
----------------------------------------
Missing wav files: 0
Duplicate pairs: 1002
Self pairs: 0


In [13]:
durations = [3, 5, 7]
dir = r"D:\my_project\SLP301-data\test_data\test_set_O"
for d in durations:
    test_list = os.path.join(dir, f"test_list_gt_{d}s.csv")
    wav_folder = os.path.join(dir, f"wav_{d}s")
    check_test_list(test_list, wav_folder)

TEST LIST REPORT
Total trials: 9879
Same speaker (1): 1701
Different speaker (0): 8178
Same ratio: 0.172
----------------------------------------
Missing wav files: 0
Duplicate pairs: 1046
Self pairs: 4490
TEST LIST REPORT
Total trials: 9876
Same speaker (1): 2093
Different speaker (0): 7783
Same ratio: 0.212
----------------------------------------
Missing wav files: 0
Duplicate pairs: 425
Self pairs: 2538
TEST LIST REPORT
Total trials: 9863
Same speaker (1): 2525
Different speaker (0): 7338
Same ratio: 0.256
----------------------------------------
Missing wav files: 0
Duplicate pairs: 1076
Self pairs: 1375
